# Tucker / HOSVD 基礎実装

このNotebookでは、**TensorLyを使う前に、PyTorchだけでHOSVD/Tucker分解の基本処理を自作する**。

目的は、mode-\(n\) unfolding / folding、mode-\(n\) product、factor matrix、core tensor、再構成の関係を自分で実装して確認すること。

`hosvd(X, ranks)` は `ranks` のkeyで分解対象modeを指定できる汎用形にする。全modeを指定すれば通常のtruncated HOSVD、modeの一部だけを指定すればpartial HOSVDとして扱う。

自作後は、完全rankでの再構成、低rankでの再構成誤差、factor matrixの直交性を確認し、TensorLyの対応APIとも照合する。

> TensorLyをPyTorch Tensorのまま使う場合は `tensorly.set_backend("pytorch")` を指定できる。


## 1. 分解対象テンソル

この3階テンソル `X` をTucker/HOSVDの自作実装に使用する。


In [1]:
import torch
from nn_compression.compression import (
    truncated_svd
)

X = torch.tensor(
    [
        [[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0]],
        [[2.0, 1.0], [4.0, 3.0], [6.0, 5.0], [8.0, 7.0]],
        [[1.0, 3.0], [2.0, 4.0], [3.0, 5.0], [4.0, 6.0]],
    ],
    dtype=torch.float32,
)

print(X)
print("shape:", X.shape)


tensor([[[1., 2.],
         [3., 4.],
         [5., 6.],
         [7., 8.]],

        [[2., 1.],
         [4., 3.],
         [6., 5.],
         [8., 7.]],

        [[1., 3.],
         [2., 4.],
         [3., 5.],
         [4., 6.]]])
shape: torch.Size([3, 4, 2])


## 2. mode-\(n\) unfolding


In [2]:
def unfold(X:torch.Tensor, mode:int)->torch.Tensor:
    """
    目的: 指定したmodeを基準に、テンソルを2次元行列へ展開する。
    X: 展開するテンソル
    mode: 展開の基準にする軸番号
    """

    if not 0 <= mode < X.ndim:
        raise ValueError(
            f"mode={mode} は 0〜{X.ndim - 1} の範囲で指定してください。"
    )
    # movedim(source, destination)はsourceをdestinationnにうつす
    return X.movedim(mode,0).flatten(start_dim=1)

### ライブラリでは

TensorLyには同じ目的の `tensorly.base.unfold` が用意されている。

```python
import tensorly as tl
tl.set_backend("pytorch")

X_mode0 = tl.unfold(X, mode=0)
```

PyTorchには **mode-\(n\) unfolding専用の同等APIはない**。`torch.Tensor.unfold` はスライディングウィンドウを取り出す別の処理なので、ここでいうunfoldingとは異なる。


## 3. folding


In [3]:
def fold(unfolded:torch.Tensor, mode:int, shape:tuple[int,...])->torch.Tensor:
    """
    目的: mode-n unfoldingされた2次元行列を、元の軸配置を持つテンソルへ戻す。

    unfolded: mode-n unfoldingされた2次元行列
    mode: unfolding時に基準とした軸番号
    shape: 復元したい元テンソルのshape
    """
    if not 0 <= mode < len(shape):
        raise ValueError(
            f"mode={mode} は 0〜{len(shape) - 1} の範囲で指定してください。"
        )
    #shape から mode 番目を一度抜いて、先頭に付け直している式
    moved_shape=(shape[mode],)+shape[:mode]+shape[mode + 1:]

    # 並びだけが違うが全体としてはもとと等しいテンソルに戻す
    X_moved = unfolded.reshape(*moved_shape)
    
    # X_movedは展開軸が最初にあるので、それを元に戻す
    return X_moved.movedim(0,mode)

### 確認

各modeで次が成立することを確認する。

\[
\mathrm{fold}(\mathrm{unfold}(X,n),n,\mathrm{shape}(X)) = X
\]

### ライブラリでは

TensorLyには `tensorly.base.fold` が用意されている。

```python
X_mode0 = tl.unfold(X, mode=0)
X_restored = tl.fold(X_mode0, mode=0, shape=X.shape)
```

PyTorchにはmode-\(n\) folding専用の同等APIはない。


## 4. mode-\(n\) product


In [ ]:
def mode_dot(X:torch.Tensor, matrix:torch.Tensor, mode:int)->torch.Tensor:
    """
    目的: テンソルの指定したmodeに2次元行列を作用させる。

    X: mode積を行うテンソル
    matrix: 指定したmodeに作用させる2次元行列
    matrix.shape == (新しいmodeのサイズ, X.shape[mode])
    mode: 行列を作用させる軸番号
    """
    
    if not 0 <= mode < len(X.shape):
        raise ValueError(
            f"mode={mode} は 0〜{len(X.shape) - 1} の範囲で指定してください。"
        )
    
    if not 2 == len(matrix.shape):
        raise ValueError(
            f"matrixは2次元行列で指定してください。"
        )
    
    if matrix.shape[1] != X.shape[mode]:
        raise ValueError(
            f"matrix.shape[1]={matrix.shape[1]} と "
            f"X.shape[{mode}]={X.shape[mode]} は一致する必要があります。"
        )

    # matrixを左から掛ける
    result_unfold = torch.matmul(matrix, unfold(X,mode))

    # 出力shapeを作る
    result_shape=X.shape[:mode]+(matrix.shape[0],)+X.shape[mode + 1:]

    return fold(result_unfold,mode,result_shape)

    
    
    


### ライブラリでは

TensorLyには `tensorly.tenalg.mode_dot` が用意されている。

```python
from tensorly.tenalg import mode_dot

Y = mode_dot(X, matrix, mode=0)
```

PyTorchにはn-mode product専用の同等APIはない。一般的なテンソル縮約には `torch.tensordot` や `torch.einsum` がある。


## 5. truncated HOSVD / partial HOSVD


In [ ]:
def hosvd(X:torch.Tensor, ranks:dict[int,int])->tuple[torch.Tensor, dict[int,torch.Tensor]]:
    """
    目的: ranksで指定したmodeごとに低rank近似を行い、
          Tucker分解に必要なcore tensorとfactor matrixを求める。

    X: Tucker分解するテンソル
    ranks: {mode: rank} 形式の辞書。
           keyが分解対象のmode、valueがそのmodeで残すrankを表す。

    返り値:
        core: 指定したmodeを低rank化したcore tensor
        factors: {mode: factor_matrix} 形式の辞書
    """
    factors:dict[int,torch.Tensor]={}
    core=X
    for mode,rank in ranks.items():
        factors[mode],_,_ = truncated_svd(unfold(X,mode),rank)
        # truncated_svd が返す Uのshapeは、[:, :rank]
        # 元のmodeの大きさを縮めるには、Tをかける
        core=mode_dot(core,factors[mode].T,mode)
    return core,factors

`ranks` に全modeを含めればtruncated HOSVD、一部のmodeだけを含めればpartial HOSVDとして使う。

```python
# 全mode
full_ranks = {0: 3, 1: 4, 2: 2}

# mode 0, 1だけ
partial_ranks = {0: 2, 1: 2}
```

factor matrixもmodeとの対応が失われないよう、`{mode: factor_matrix}` の形で扱う。


### ライブラリでは

PyTorchにはHOSVDそのものを行う関数はなく、行列SVDの `torch.linalg.svd` が自作時の基本部品になる。

```python
U, S, Vh = torch.linalg.svd(matrix, full_matrices=False)
```

TensorLyには全modeのTucker分解を行う `tensorly.decomposition.tucker` と、指定modeだけを分解する `partial_tucker` がある。

```python
from tensorly.decomposition import tucker, partial_tucker

# 全mode
core_tl, factors_tl = tucker(
    X,
    rank=[2, 2, 2],
    init="svd",
)

# mode 0, 1だけ
modes = [0, 1]
core_partial, factors_partial = partial_tucker(
    X,
    rank=[2, 2],
    modes=modes,
    init="svd",
)

# TensorLyはfactorをlistで返すため、mode対応を明示したい場合
factors_partial_by_mode = dict(zip(modes, factors_partial))
```

**注意:** TensorLyの `tucker` / `partial_tucker` はHigher Order Orthogonal Iteration (HOI/HOOI)によるTucker分解で、自作する一回のtruncated HOSVDと完全に同じ処理ではない。比較ではfactorの値の完全一致ではなく、shape・再構成結果・再構成誤差を確認する。


## 6. Tucker再構成


In [7]:
def reconstruct_tucker(
    core: torch.Tensor,
    factors: dict[int, torch.Tensor],
) -> torch.Tensor:
    """
    core tensorとfactor matrixから、元テンソルの近似を再構成する。
    全mode分解と部分mode分解の両方を扱う。
    """
    original = core
    for mode, U in factors.items():
        original = mode_dot(original, U, mode)

    return original

### ライブラリでは

全modeの通常のTucker分解なら、TensorLyの `tensorly.tucker_tensor.tucker_to_tensor` を使える。

```python
from tensorly.tucker_tensor import tucker_to_tensor

X_hat_tl = tucker_to_tensor((core_tl, factors_tl))
```

部分modeだけを分解した `partial_tucker` の結果は、指定したmodeにfactorを掛け戻せば再構成できる。

```python
from tensorly.tenalg import multi_mode_dot

X_hat_partial = multi_mode_dot(
    core_partial,
    factors_partial,
    modes=modes,
)
```

PyTorchにはTucker形式の `core + factors` を直接受け取って再構成する専用APIはない。


## 7. 再構成誤差


In [8]:
def relative_frobenius_error(
    X: torch.Tensor,
    X_hat: torch.Tensor,
) -> torch.Tensor:
    """
    元テンソルと再構成テンソルの相対Frobenius誤差を返す。

    ||X - X_hat||_F / ||X||_F
    """
    if X.shape != X_hat.shape:
        raise ValueError(
            f"shapeが一致しません: X={tuple(X.shape)}, "
            f"X_hat={tuple(X_hat.shape)}"
        )

    # 分母になるXの大きさ計算
    denominator = torch.linalg.vector_norm(X)

    if denominator == 0:
        raise ValueError("Xがゼロテンソルなので相対誤差を定義できません。")

    return torch.linalg.vector_norm(X - X_hat) / denominator


完全rankで全modeをHOSVDした場合は、数値誤差を除いて再構成誤差が十分小さくなることを確認する。

\[
\frac{\|X-\hat X\|_F}{\|X\|_F}
\]

PyTorchではFrobenius normの計算に `torch.linalg.vector_norm` などを利用できる。


## 8. factor matrixの直交性


In [9]:
def orthogonality_error(U: torch.Tensor) -> torch.Tensor:
    """
    factor matrix U の列がどの程度直交規格化されているかを返す。

    完全に直交規格化されていれば 0 に近い値になる。
    """
    if U.ndim != 2:
        raise ValueError("Uは2次元行列で指定してください。")

    identity = torch.eye(
        U.shape[1],
        dtype=U.dtype,
        device=U.device,
    )

    return torch.linalg.matrix_norm(U.T @ U - identity, ord="fro")


HOSVDで得たfactor matrixについて、

\[
U^\mathsf{T}U \simeq I
\]

となることを確認する。

ここでも行列積やnorm自体はPyTorchの基本演算を使ってよい。


## 9. 自作実装の確認

次の順で確認する。

1. 各modeで `fold(unfold(X, mode), mode, X.shape)` が `X` に戻る
2. 全modeを完全rankでHOSVDしたとき再構成誤差が十分小さい
3. rankを下げると再構成誤差が変化する
4. 一部modeだけを指定したpartial HOSVDでも元shapeへ再構成できる
5. `factors` のkeyと元テンソルのmodeの対応が保たれている
6. 各factor matrixの直交性を確認する


In [22]:
print(f"Ans1:{fold(unfold(X,1),1,X.shape)}")
"------------------------------"
rank:dict[int,int]={
    0:3,
    1:4,
    2:2
}
print(f"Ans2:{relative_frobenius_error(X,reconstruct_tucker(*hosvd(X,rank)))}")
"------------------------------"
rank:dict[int,int]={
    0:2,
    1:1,
    2:2
}
print(f"Ans3:{relative_frobenius_error(X,reconstruct_tucker(*hosvd(X,rank)))}")
"------------------------------"
rank:dict[int,int]={
    0:3,
    2:2
}
print(f"Ans4:{reconstruct_tucker(*hosvd(X,rank))}")
"------------------------------"
_,factors=hosvd(X,rank)
for mode,U in factors.items():
    print(f"Ans6:mode->{mode},orthogonality_error->{orthogonality_error(U)}")

Ans1:tensor([[[1., 2.],
         [3., 4.],
         [5., 6.],
         [7., 8.]],

        [[2., 1.],
         [4., 3.],
         [6., 5.],
         [8., 7.]],

        [[1., 3.],
         [2., 4.],
         [3., 5.],
         [4., 6.]]])
Ans2:2.2447464687047614e-07
Ans3:0.0880948156118393
Ans4:tensor([[[1.0000, 2.0000],
         [3.0000, 4.0000],
         [5.0000, 6.0000],
         [7.0000, 8.0000]],

        [[2.0000, 1.0000],
         [4.0000, 3.0000],
         [6.0000, 5.0000],
         [8.0000, 7.0000]],

        [[1.0000, 3.0000],
         [2.0000, 4.0000],
         [3.0000, 5.0000],
         [4.0000, 6.0000]]])
Ans6:mode->0,orthogonality_error->1.884864389012364e-07
Ans6:mode->2,orthogonality_error->3.3717478231665154e-07


## 10. TensorLyとの照合

自作実装が完成したら、同じテンソルをTensorLyでも処理し、次を比較する。

- unfolding / foldingのshapeと値
- full / partialでのcore tensorのshape
- factor matrixのshapeとmode対応
- 再構成テンソル
- 再構成誤差
- factor matrixの直交性

factor matrixそのものは符号や基底の取り方が異なる場合があるため、要素ごとの完全一致だけを正解条件にはしない。


In [39]:
import tensorly as tl
from tensorly.decomposition import tucker, partial_tucker
from tensorly.tucker_tensor import tucker_to_tensor
from tensorly.tenalg import multi_mode_dot

tl.set_backend("pytorch")

# --- unfolding / folding の shape と値 ---
for mode in range(X.ndim):
    X_unf = unfold(X, mode)
    X_unf_tl = tl.unfold(X, mode)
    X_fold = fold(X_unf, mode, X.shape)
    X_fold_tl = tl.fold(X_unf_tl, mode, X.shape)
    print(
        f"mode={mode}: unfold close={torch.allclose(X_unf, X_unf_tl)}, "
        f"fold close={torch.allclose(X_fold, X_fold_tl)}, "
        f"unfold shape={tuple(X_unf.shape)}"
    )

# --- full: core / factors の shape、再構成、誤差、直交性 ---
full_ranks = {0: 2, 1: 2, 2: 2}
core, factors = hosvd(X, full_ranks)
X_hat = reconstruct_tucker(core, factors)

core_tl, factors_tl = tucker(X, rank=[2, 2, 2], init="svd")
X_hat_tl = tucker_to_tensor((core_tl, factors_tl))

print("full core shape: self=", tuple(core.shape), "tl=", tuple(core_tl.shape))
for mode, U in factors.items():
    print(
        f"full factor mode={mode}: self={tuple(U.shape)}, "
        f"tl={tuple(factors_tl[mode].shape)}, "
        f"orth self={orthogonality_error(U).item():.3e}, "
        f"tl={orthogonality_error(factors_tl[mode]).item():.3e}"
    )
print(
    "full recon error: self=",
    relative_frobenius_error(X, X_hat).item(),
    "tl=",
    relative_frobenius_error(X, X_hat_tl).item(),
)

# --- partial: core shape、mode対応、再構成 ---
modes = [0, 1]
partial_ranks = {0: 2, 1: 2}
core_p, factors_p = hosvd(X, partial_ranks)
X_hat_p = reconstruct_tucker(core_p, factors_p)

(core_partial, factors_partial), _ = partial_tucker(
    X,
    rank=[2, 2],
    modes=modes,
    init="svd",
)
factors_partial_by_mode = dict(zip(modes, factors_partial))
X_hat_partial = multi_mode_dot(core_partial, factors_partial, modes=modes)

print("partial core shape: self=", tuple(core_p.shape), "tl=", tuple(core_partial.shape))
print("partial factor keys: self=", sorted(factors_p), "tl=", sorted(factors_partial_by_mode))
for mode in modes:
    print(
        f"partial factor mode={mode}: self={tuple(factors_p[mode].shape)}, "
        f"tl={tuple(factors_partial_by_mode[mode].shape)}"
    )
print(
    "partial recon error: self=",
    relative_frobenius_error(X, X_hat_p).item(),
    "tl=",
    relative_frobenius_error(X, X_hat_partial).item(),
)

mode=0: unfold close=True, fold close=True, unfold shape=(3, 8)
mode=1: unfold close=True, fold close=True, unfold shape=(4, 6)
mode=2: unfold close=True, fold close=True, unfold shape=(2, 12)
full core shape: self= (2, 2, 2) tl= (2, 2, 2)
full factor mode=0: self=(3, 2), tl=(3, 2), orth self=1.738e-07, tl=2.384e-07
full factor mode=1: self=(4, 2), tl=(4, 2), orth self=3.863e-07, tl=4.787e-07
full factor mode=2: self=(2, 2), tl=(2, 2), orth self=3.372e-07, tl=1.686e-07
full recon error: self= 0.03588838502764702 tl= 0.03588837385177612
partial core shape: self= (2, 2, 2) tl= (2, 2, 2)
partial factor keys: self= [0, 1] tl= [0, 1]
partial factor mode=0: self=(3, 2), tl=(3, 2)
partial factor mode=1: self=(4, 2), tl=(4, 2)
partial recon error: self= 0.03588838502764702 tl= 0.03588838502764702
